In [1]:
# ---------------------------------------------------------
# BRONZE INGESTION STEP (RAW FILES, KEEP FULL + SUMMARY)
# Purpose: Download Airbnb datasets into Bronze with clear
# naming to avoid overwriting:
#   - listings_full.csv.gz → listings_full.csv
#   - reviews_full.csv.gz → reviews_full.csv
#   - calendar.csv.gz → calendar.csv
#   - listings_summary.csv
#   - reviews_summary.csv
#   - neighbourhoods.csv
#   - neighbourhoods.geojson
# ---------------------------------------------------------

import requests, os, gzip, shutil

snapshot_date = "2025-03-04"  # <-- using the snapshot you picked
base_data_url = f"http://data.insideairbnb.com/united-kingdom/england/london/{snapshot_date}/data"
base_vis_url  = f"http://data.insideairbnb.com/united-kingdom/england/london/{snapshot_date}/visualisations"

files = {
    "listings_full": f"{base_data_url}/listings.csv.gz",
    "calendar": f"{base_data_url}/calendar.csv.gz",
    "reviews_full": f"{base_data_url}/reviews.csv.gz",
    "listings_summary": f"{base_vis_url}/listings.csv",
    "reviews_summary": f"{base_vis_url}/reviews.csv",
    "neighbourhoods": f"{base_vis_url}/neighbourhoods.csv",
    "neighbourhoods_geo": f"{base_vis_url}/neighbourhoods.geojson"
}

os.makedirs("data/bronze", exist_ok=True)

for name, url in files.items():
    # Decide file extension
    if url.endswith(".gz"):
        raw_path = f"data/bronze/{name}.csv.gz"
    elif url.endswith(".geojson"):
        raw_path = f"data/bronze/{name}.geojson"
    else:
        raw_path = f"data/bronze/{name}.csv"

    # Download
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        print(f"❌ Failed to fetch {url} (status {r.status_code})")
        continue

    with open(raw_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    # If compressed, also decompress to .csv
    if raw_path.endswith(".csv.gz"):
        csv_path = raw_path.replace(".csv.gz", ".csv")
        with gzip.open(raw_path, "rb") as f_in, open(csv_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f"✅ Saved {name} → {raw_path} (compressed) and {csv_path} (decompressed)")
    else:
        print(f"✅ Saved {name} → {raw_path}")

✅ Saved listings_full → data/bronze/listings_full.csv.gz (compressed) and data/bronze/listings_full.csv (decompressed)
✅ Saved calendar → data/bronze/calendar.csv.gz (compressed) and data/bronze/calendar.csv (decompressed)
✅ Saved reviews_full → data/bronze/reviews_full.csv.gz (compressed) and data/bronze/reviews_full.csv (decompressed)
✅ Saved listings_summary → data/bronze/listings_summary.csv
✅ Saved reviews_summary → data/bronze/reviews_summary.csv
✅ Saved neighbourhoods → data/bronze/neighbourhoods.csv
✅ Saved neighbourhoods_geo → data/bronze/neighbourhoods_geo.geojson
